<b><font size="6" color="#E8800A">Week 6 · Linear and logistic models</font></b><br>
<b><font size="4">One idea, two targets, and two very different lessons</font></b><br>

This week builds the course's first two **baselines**: logistic
regression for a categorical target, then linear regression for a numeric one.
Both compute a weighted sum of the inputs, and the type of the target decides
what they do with it. The classification half comes first, because your project
this year is a classification task.

We read the weights twice, from scikit-learn and from **statsmodels**, which adds
a standard error, an interval and a p-value to each. Then every setting is
changed alone against the defaults, one grid search per task fits every
combination of them, and nested cross-validation says whether the search was
worth running. On `champions` it was not: the search's nested estimate sits below
the default, although its winner scores higher on the test rows. On `cars4you`
changing the target pays and no hyperparameter does.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Two models, one idea</font>](#one-idea)
* [<font color='#E8800A'>What goes wrong on a class label</font>](#wrong)
* [<font color='#E8800A'>Predicting a class: logistic regression</font>](#logistic)
* [<font color='#E8800A'>Did the right model win?</font>](#compare)
* [<font color='#E8800A'>Reading a logistic model</font>](#odds)
* [<font color='#E8800A'>Alternatives to the default penalty: L2, L1 and elastic net</font>](#logistic-params)
* [<font color='#E8800A'>Nested cross-validation on the athletes</font>](#nested-athletes)
* [<font color='#E8800A'>Predicting a number: linear regression</font>](#linear)
* [<font color='#E8800A'>Reading coefficients without being fooled</font>](#coefficients)
* [<font color='#E8800A'>Alternatives to least squares: Ridge, Lasso and Elastic Net</font>](#linear-params)
* [<font color='#E8800A'>Transforming the target: log(price)</font>](#target)
* [<font color='#E8800A'>Nested cross-validation on the cars</font>](#nested-cars)
* [<font color='#E8800A'>Two numbers we stand behind</font>](#two-numbers)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Two models, one idea</font> <a class="anchor" id="one-idea"></a>
[Back to TOC](#toc)

Both models start from the same object, a **weighted sum** of the
features, one weight per column plus an intercept:

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p .$$

Linear regression stops there and reports $z$ as the prediction. Logistic
regression passes $z$ through a function that squashes it into $(0, 1)$ and
reports the result as a probability.

<div class="alert alert-block alert-info">

**The target's type decides which model you may use, and that is
settled before any score is computed.**

| target | model | what it predicts | range of the prediction |
|---|---|---|---|
| continuous (a price, a temperature) | linear regression | the conditional mean of $y$ | $(-\infty, +\infty)$ |
| binary (yes / no) | logistic regression | $P(y = 1 \mid x)$ | $(0, 1)$ |
| more than two classes | multinomial logistic | one probability per class, summing to 1 | $(0, 1)$ each |

A price has no fixed upper end, so an unbounded prediction is a small problem
there. A probability cannot exceed 1 or fall below 0, so a model whose output is
unbounded is predicting things that cannot happen.

**What linear regression assumes**, and what you should be prepared to defend:
the relationship is additive and linear in the coefficients; the errors have
roughly constant spread; and observations are independent. **What logistic
regression assumes**: the same additivity, but on the *log-odds* scale rather
than on $y$ itself.

**The caveat.** Matching the model to the target makes the output *meaningful*.
It does not promise a better score, or that any setting will help.

</div>

In [ ]:
import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RepeatedKFold,
    RepeatedStratifiedKFold,
    cross_validate,
    train_test_split,
)

from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, result_context

random_state = 33
np.random.seed(random_state)  # For reproducibility
figsize = (7, 5)

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)
# scikit-learn 1.8 recommends C=np.inf instead of the deprecated penalty=None,
# and its compatibility path still emits this legacy message for that exact
# representation.
warnings.filterwarnings(
    "ignore",
    message="Setting penalty=None will ignore the C and l1_ratio parameters",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


# <font color='#E8800A'>What goes wrong on a class label</font> <a class="anchor" id="wrong"></a>
[Back to TOC](#toc)

The classification spine is `champions`: `Outcome` is 0 or 1, a
category stored as a number. Nothing stops you handing that to
`LinearRegression`; it is numeric, the code runs, and you get predictions.

The argument against doing it was made in the first section, from the range of
the output. Let us see it happen.

__Step 1:__ Load the classification spine and split it.

In [ ]:
champions, classification_recipe = load_recipe(
    "../../data/interim/champions.csv",
    classification_recipe_log,
)
# RecordID names each row, so it becomes the index.
champions = champions.set_index("RecordID")
X_churn = champions.loc[:, classification_recipe["categorical"] + classification_recipe["numeric"]].copy()
y_churn = champions["Outcome"]

X_churn_train, X_churn_test, y_churn_train, y_churn_test = train_test_split(
    X_churn, y_churn, test_size=0.2, stratify=y_churn, random_state=random_state,
)
print(y_churn.value_counts().to_dict(),
      f"-- positive rate {y_churn.mean():.4f}")

**2,416 positives against 1,584 negatives**, a positive rate of
**0.6040**. Stratifying keeps that ratio in both halves.

__Step 2:__ Prepare the athletes, and fix the folds every score is read on.

In [ ]:
churn_prep = FoldPreprocessor(classification_recipe).fit(X_churn_train, y_churn_train)
X_churn_train_ready = churn_prep.transform(X_churn_train)
X_churn_test_ready = churn_prep.transform(X_churn_test)

# Ten folds of the training athletes: every validation score below is a mean over them.
churn_folds = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=random_state)
churn_prepared = prepared_folds(
    FoldPreprocessor(classification_recipe), churn_folds,
    X_churn_train, y_churn_train, n_jobs=4)

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html'>sklearn.linear_model.LinearRegression(*, fit_intercept=True)</a>

**Definition:**
Ordinary least squares: the weights that minimise the squared error between the weighted sum and the target, with no penalty.

**Main Parameters:**
- `fit_intercept` (default `True`): whether the weighted sum gets a constant term.

**Main Attributes (after fitting):**
- `coef_`: one weight per feature, in target units per unit of the column.
- `intercept_`: the constant term.
- `rank_`: how many columns the fit could separate.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`: the weighted sum itself, with no bound on its range.
</div>

__Step 3:__ Fit a linear regression to the 0/1 target, check what it predicts, and score it.

In [ ]:
wrong_folds = [
    LinearRegression().fit(fold["X_fit"], fold["y_fit"]).predict(fold["X_score"])
    for fold in churn_prepared
]
wrong_predictions = np.concatenate(wrong_folds)

impossible = (wrong_predictions < 0) | (wrong_predictions > 1)
print(f"predicted values run from {wrong_predictions.min():.3f}"
      f" to {wrong_predictions.max():.3f}")
print(f"outside [0, 1]: {impossible.sum()} of {len(wrong_predictions)}"
      f" validation predictions ({impossible.mean():.1%})")

In [ ]:
# Where the impossible values sit, against the only two values the
# target can actually take.
plt.figure(figsize=figsize)
plt.scatter(range(len(wrong_predictions)), wrong_predictions, s=6,
            c=[PLOT_ORANGE if bad else PLOT_BLUE for bad in impossible])
plt.axhline(0, color="black", linewidth=0.8)
plt.axhline(1, color="black", linewidth=0.8)
plt.ylabel("predicted 'probability' of the positive outcome")
plt.xlabel("validation prediction, fold by fold")
plt.title("A linear model does not know the target is bounded")
plt.show()

In [ ]:
# Cut at 0.5, its predictions become labels, scored like any classifier's.
wrong_labels = [(predictions >= 0.5).astype(int) for predictions in wrong_folds]
wrong_test = (LinearRegression().fit(X_churn_train_ready, y_churn_train)
              .predict(X_churn_test_ready) >= 0.5).astype(int)
for name, metric in [("F1", f1_score), ("accuracy", accuracy_score)]:
    validation = np.mean([metric(fold["y_score"], labels)
                          for fold, labels in zip(churn_prepared, wrong_labels)])
    print(f"{name:8s} validation {validation:.4f}, test {metric(y_churn_test, wrong_test):.4f}")

<div class="alert alert-block alert-warning">

**The validation predictions run from -0.812 to 1.557, and 710 of
the 6,400, 11.1 %, fall outside the interval the target lives in.** There is no
error and no warning. Read as probabilities, these tell a stakeholder that one
athlete has a **155.7 %** chance of the outcome and another a negative one. Cut
at 0.5 the same predictions score F1 **0.8499** on the folds and **0.8485** on
the test rows, so the score gives no warning either.

The problem is structural: a weighted sum is unbounded, so no choice of
coefficients keeps it inside $[0, 1]$ for every possible input. Predictions that
happen to land inside on these rows carry no guarantee for the next ones.

</div>

# <font color='#E8800A'>Predicting a class: logistic regression</font> <a class="anchor" id="logistic"></a>
[Back to TOC](#toc)

Logistic regression keeps the weighted sum and adds one step: it passes
$z$ through a function whose output cannot leave $(0, 1)$, no matter what $z$
is.

<div class="alert alert-block alert-info">

**The sigmoid, and where the name comes from.** The squashing
function is

$$\sigma(z) = \frac{1}{1 + e^{-z}},$$

which tends to 0 as $z \to -\infty$ and to 1 as $z \to +\infty$, never
reaching either. So $P(y=1 \mid x) = \sigma(z)$ is a probability by
construction rather than by luck.

Turn it around and you get the reason for the name. If $p = \sigma(z)$ then

$$z = \log\!\left(\frac{p}{1-p}\right),$$

the **log-odds**, or *logit*, of $p$. So logistic regression is a linear model:
it is linear in the log-odds instead of in $y$. That is the assumption it asks
you to defend.

</div>

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html'>sklearn.linear_model.LogisticRegression(*, C=1.0, fit_intercept=True, l1_ratio=0.0, class_weight=None, solver='lbfgs', max_iter=100)</a>

**Definition:**
Linear classifier fitting the log-odds of the positive class, with a penalty on the coefficients by default.

**Main Parameters:**
- `C` (default `1.0`): **inverse** penalty strength. Small `C` means a heavy penalty; `C=np.inf` removes it.
- `l1_ratio` (default `0.0`): the penalty's shape, from L2 at 0 to L1 at 1.
- `class_weight` (default `None`): `'balanced'` reweights each class by the inverse of its frequency.
- `fit_intercept` (default `True`): whether the log-odds get a constant term.
- `solver` (default `'lbfgs'`): the optimiser; `'saga'` for L1 and elastic net.
- `max_iter` (default `100`): iteration budget; raise it when the solver warns it did not converge.

**Main Attributes (after fitting):**
- `coef_`: one weight per feature, on the log-odds scale.
- `intercept_`: the constant term.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`: labels at the 0.5 threshold.
- `predict_proba(X)`: the probabilities themselves.
</div>

__Step 4:__ Fit the model the target type actually calls for.

In [ ]:
# The model whose odds ratios are read below.
logistic = LogisticRegression(max_iter=1000).fit(X_churn_train_ready, y_churn_train)
# The same model on each fold, keeping the probability of class 1 for the
# validation part.
probability_folds = [
    LogisticRegression(max_iter=1000).fit(fold["X_fit"], fold["y_fit"])
    .predict_proba(fold["X_score"])[:, 1]
    for fold in churn_prepared
]
churn_probabilities = np.concatenate(probability_folds)

print(f"probabilities run from {churn_probabilities.min():.4f}"
      f" to {churn_probabilities.max():.7f}")
print("outside (0, 1):",
      ((churn_probabilities <= 0) | (churn_probabilities >= 1)).sum())

**0.0001 to 0.9994133 over the same 6,400 validation predictions,
and nothing outside the interval.** The model comes within a thousandth of
certainty at the top and reaches neither end, because $\sigma$ cannot produce 0
or 1. That guarantee is in the model, not in the data, so it holds on next
season's entries too. Whether a predicted 0.8 comes true eight times in ten is a
separate property, **calibration**, and the bound does not promise it.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html'>sklearn.dummy.DummyClassifier(*, strategy='prior', random_state=None, constant=None)</a>

**Definition:**
A classifier that ignores the features and predicts from the training labels alone: the floor any real model has to beat.

**Main Parameters:**
- `strategy` (default `'prior'`): the rule; `'most_frequent'` labels every row with the training majority.

**Main Attributes (after fitting):**
- `class_prior_`: the share of each class in the training labels.

**Main Methods:**
- `fit(X, y)`: reads only `y`.
- `predict(X)`
</div>

__Step 5:__ Score a model that always predicts the majority, and open the log.

In [ ]:
# The floor labels every athlete with the majority outcome.
churn_floor_cv = cross_validate(
    DummyClassifier(strategy="most_frequent"), X_churn_train, y_churn_train,
    cv=churn_folds, scoring=["f1", "recall", "balanced_accuracy"], return_train_score=True)
churn_floor = DummyClassifier(strategy="most_frequent").fit(X_churn_train, y_churn_train)

In [ ]:
churn_log = []
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "always the majority",
                   "train": churn_floor_cv["train_f1"],
                   "validation": churn_floor_cv["test_f1"], "model": churn_floor},
       X_churn_test, y_churn_test, metric=f1_score, label="F1")
floor_labels = churn_floor.predict(X_churn_test)
print(f"\nrecall:            validation {churn_floor_cv['test_recall'].mean():.4f},"
      f" test {recall_score(y_churn_test, floor_labels):.4f}")
print(f"balanced accuracy: validation {churn_floor_cv['test_balanced_accuracy'].mean():.4f},"
      f" test {balanced_accuracy_score(y_churn_test, floor_labels):.4f}")

The log gives every model one row: its mean training and validation
F1 over the ten folds, then its test F1. The validation column chooses; the test
column only records.

__Step 6:__ Score the logistic regression on the same folds, and log it.

In [ ]:
logistic_model = PreparedEstimator(FoldPreprocessor(classification_recipe),
                                   ...)
logistic_cv = cross_validate(
    logistic_model, X_churn_train, y_churn_train, cv=churn_folds,
    scoring=["f1", "recall", "balanced_accuracy"], return_train_score=True)
logistic_fitted = clone(logistic_model).fit(X_churn_train, y_churn_train)

In [ ]:
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "LogisticRegression(C=1)",
                   "train": logistic_cv["train_f1"],
                   "validation": ..., "model": logistic_fitted},
       X_churn_test, y_churn_test, metric=f1_score, label="F1")
logistic_labels = logistic_fitted.predict(X_churn_test)
print(f"\nrecall:            validation {logistic_cv['test_recall'].mean():.4f},"
      f" test {recall_score(y_churn_test, logistic_labels):.4f}")
print(f"balanced accuracy: validation {logistic_cv['test_balanced_accuracy'].mean():.4f},"
      f" test {balanced_accuracy_score(y_churn_test, logistic_labels):.4f}")

F1 **0.8494** on the validation folds and **0.8474** on the test
rows, against the floor's **0.7532** and **0.7529**. The floor calls every
athlete a positive, so its **recall is 1.0000** and its **balanced accuracy
0.5000** in both columns, which is what "no information" looks like on a metric
that averages the two classes. Answering "yes" to everything scores 0.75 on F1
when 60 % of the answers are yes, so F1 alone would make the floor look
respectable. That is why every classification result in this course reports
**F1, ROC-AUC and balanced accuracy** together.

# <font color='#E8800A'>Did the right model win?</font> <a class="anchor" id="compare"></a>
[Back to TOC](#toc)

We have argued that logistic regression is the correct model for this
target, and shown that the linear one predicts impossible values. There is one
question left, and it is the uncomfortable one: **does the correct model actually
score better?**

__Step 7:__ Compare the two models like for like, on the same rows.

In [ ]:
# Both models label each fold's validation part and the test rows.
churn_labels = [(probabilities >= 0.5).astype(int) for probabilities in probability_folds]
test_labels = {
    "linear @ 0.5": wrong_test,
    "logistic": LogisticRegression(max_iter=1000).fit(
        X_churn_train_ready, y_churn_train).predict(X_churn_test_ready),
}
for name, labels in [("linear @ 0.5", wrong_labels), ("logistic", churn_labels)]:
    accuracy = np.mean([accuracy_score(fold["y_score"], fold_labels)
                        for fold, fold_labels in zip(churn_prepared, labels)])
    f1 = np.mean([f1_score(fold["y_score"], fold_labels)
                  for fold, fold_labels in zip(churn_prepared, labels)])
    print(f"{name:13s} validation accuracy={accuracy:.4f}  F1={f1:.4f}"
          f" | test accuracy={accuracy_score(y_churn_test, test_labels[name]):.4f}"
          f"  F1={f1_score(y_churn_test, test_labels[name]):.4f}")
agreement = (np.concatenate(wrong_labels) == np.concatenate(churn_labels)).mean()
print(f"the two models agree on {agreement:.1%} of validation predictions")

<div class="alert alert-block alert-warning">

**Neither model wins outright.** Over the ten folds logistic is
ahead on accuracy, **0.8097** against **0.8062**, and the linear model on F1,
**0.8499** against **0.8494**; the two agree on **97.1 %** of the validation
predictions. On the test rows they tie on accuracy at **0.8050**, and the linear
model leads on F1, **0.8485** against **0.8474**.

Resist the temptation to file either lead as a verdict. **Choosing the model
that matches your target is not a promise of a better score**, and the argument
for logistic regression never rested on the scoreboard:

1. **Its output means something.** A probability can go in a report, rank
   athletes, or carry a cost; the linear model's 1.557 cannot.
2. **The guarantee does not depend on the data.** Nothing stopped the linear
   model leaving $[0, 1]$, and 710 times it did.
3. **Both gaps are under half a point.** A lead counts only if the
   fold-by-fold differences stay on one side of zero, and even a real one would
   not make the linear model's output a probability.

</div>

# <font color='#E8800A'>Reading a logistic model</font> <a class="anchor" id="odds"></a>
[Back to TOC](#toc)

The coefficients read here are `logistic`'s. They are on the
log-odds scale, so they need converting before they can be said out loud.

__Step 8:__ Convert the coefficients into something sayable.

In [ ]:
odds_table = pd.DataFrame({
    "coefficient": logistic.coef_[0],
    "odds_ratio": np.exp(logistic.coef_[0]),
}, index=churn_prep.get_feature_names_out()).sort_values(
    "odds_ratio", ascending=False)
print(pd.concat([odds_table.head(4), odds_table.tail(3)]).round(4).to_string())

<div class="alert alert-block alert-info">

**How to say these out loud.** A coefficient $\beta_j$ is the
change in log-odds per one unit of feature $j$, holding the other columns fixed.
Its exponential is an **odds ratio**, which multiplies the odds: above 1 the odds
of the positive outcome rise with the feature, below 1 they fall.

The column decides what one unit is: **one interquartile range** for the
`RobustScaler` numeric columns, and the **whole step** from the dropped level
for an indicator. So `Competition_Olympic Games` at **18.7401** gives an athlete
at the Olympic Games about nineteen times the odds of an otherwise identical
athlete at a Continental Championship, the level `drop="first"` removed.

**Odds are not probability.** Odds of 2 mean a probability of $2/3$, and
doubling the odds does not double the probability: near $p = 0.9$ it barely
moves it.

</div>

__Step 9:__ Fit the same model in statsmodels, which adds the statistical table scikit-learn does not produce.

scikit-learn stops at the coefficients: it has no statistical table
for a regression. **statsmodels** has one. `sm.Logit` fits the model without a
penalty, while scikit-learn's default is penalised at `C=1`, so to compare like
with like the scikit-learn side is refitted with `C=np.inf`, which removes the
penalty. The first table is what scikit-learn holds, `intercept_` and `coef_`;
the second is the table statsmodels prints, which adds to each coefficient a
standard error, a p-value for "this coefficient is zero" and a 95% interval.

In [ ]:
logit_features = pd.DataFrame(
    X_churn_train_ready, columns=churn_prep.get_feature_names_out(),
    index=X_churn_train.index,
).astype(float)

# C=np.inf removes scikit-learn's default penalty, so both fits maximise the
# same likelihood.
unpenalized_logistic = LogisticRegression(C=np.inf, max_iter=5000, tol=1e-10).fit(
    logit_features, y_churn_train)
sklearn_logit = pd.DataFrame(
    {"coefficient": np.r_[unpenalized_logistic.intercept_, unpenalized_logistic.coef_[0]]},
    index=["const", *logit_features.columns])
sklearn_logit["odds_ratio"] = np.exp(sklearn_logit["coefficient"])
sklearn_logit.round(4)

In [ ]:
logit_design = sm.add_constant(logit_features, has_constant="add")
logit_result = sm.Logit(y_churn_train.astype(float), logit_design).fit(
    maxiter=1000, disp=False)
print(logit_result.summary())

# A standard error above 10 on the log-odds scale means the rows cannot pin the
# coefficient down.
for column in logit_design.columns[logit_result.bse > 10]:
    carriers = y_churn_train[logit_features[column] == 1]
    print(f"\n{column}: {len(carriers)} training athlete(s) carry it,"
          f" outcome {carriers.tolist()}")

**Where both fits are well defined they agree** to the third
decimal. The default penalty turns out to have been shrinking the odds ratios:
unpenalised, `Competition_Olympic Games` rises from 18.7401 to **42.4479**, and
statsmodels puts its coefficient, **3.7483** in log-odds, in a 95% interval from
**2.831 to 4.665**.

**A predictor whose p-value is above 0.05 is not significant at the 5% level,
and an insignificant predictor can be removed.** Leaving `No coach_True` aside,
ten are, from `Age group_55<=` at **0.817** and `Sand training` at **0.791** down
to `Other training` at **0.117**. A small p-value says the data pin a coefficient
away from zero. It does not say the feature causes the outcome, or rank the
features by importance.

**`No coach_True` is the row the intervals exist to catch.** The penalised table
put its odds ratio at 0.4448, which reads as "under half the odds without a
coach". The line under the summary shows that **one** training athlete has no
coach, with outcome 0. The likelihood therefore keeps improving as the
coefficient heads to minus infinity: statsmodels stops at **-29.4374**,
scikit-learn at **-21.0275**, and the standard error is astronomical. The
summary's `converged: False` says the optimisation failed, and more iterations
would not help, because there is no maximum to reach. The data cannot estimate
this coefficient, and the 0.4448 is the penalty's number, not the data's.

# <font color='#E8800A'>Alternatives to the default penalty: L2, L1 and elastic net</font> <a class="anchor" id="logistic-params"></a>
[Back to TOC](#toc)

`LogisticRegression` charges its weights a penalty by default: the L2
penalty, at a strength that falls as `C` rises. Its alternatives change the
penalty's shape, or remove it.

| penalty | set with | what it does to the coefficients |
|---|---|---|
| L2, the default | `l1_ratio=0` | shrinks every coefficient towards zero |
| L1 | `l1_ratio=1`, `solver="saga"` | shrinks, and sets the weakest exactly to zero |
| elastic net | `0 < l1_ratio < 1`, `solver="saga"` | both, mixed by `l1_ratio` |
| none | `C=np.inf` | nothing is charged |

scikit-learn 1.8 replaced the `penalty` argument with `l1_ratio`, and only the
`saga` solver fits the L1 and elastic-net penalties.

Each step below changes **one** setting and holds every other at its
default, prints validation and test F1 side by side, and logs its best value.
The steps are here to show what each setting does; the choice is made by one grid
search over every combination of the settings tested, at the end of the section.
In a project you would go straight to that grid.

**`C`: how much the model is charged for large weights**, with
the scale running backwards. `C` is the *inverse* of the penalty strength, so
`C=100` is nearly unpenalised and `C=0.001` heavily penalised. As the charge
grows the weights are pulled towards zero, and in the limit the model predicts
the same probability for every athlete.

That is a trade, not an improvement. **Prediction: a plateau for the light
penalties and a collapse at the heavy end, reached first by L1 and elastic net,
because they set weights exactly to zero.**

__Step 10:__ Walk C across five orders of magnitude for each penalty; at `C=1` each is at its defaults.

In [ ]:
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
# saga gets more iterations than its default 100, so every L1 and elastic-net
# fit converges, and a seed, because it visits the rows in a random order.
penalties = {"L2": LogisticRegression(max_iter=5000),
             "L1": LogisticRegression(solver="saga", l1_ratio=1.0, max_iter=5000,
                                      random_state=random_state),
             "elastic net": LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000,
                                               random_state=random_state)}

In [ ]:
c_rows = []
for family, default in penalties.items():
    for value in c_values:
        model = clone(default).set_params(C=value)
        fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
        validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                      for fold, fitted in zip(churn_prepared, fold_models)]
        c_rows.append({
            "configuration": f"{family}, C={value:g}", "family": family, "C": value,
            "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                      for fold, fitted in zip(churn_prepared, fold_models)],
            "validation": validation, "cv_F1": np.mean(validation),
            "test_F1": f1_score(y_churn_test, clone(model).fit(
                X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)),
            "model": model,
        })
c_sweep = pd.DataFrame(c_rows)
c_sweep.pivot(index="C", columns="family", values=["cv_F1", "test_F1"]).round(4)

In [ ]:
# The sweep's winner joins the log.
best = c_sweep.loc[c_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
figure, panels = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for panel, family in zip(panels, penalties):
    rows = c_sweep[c_sweep["family"] == family]
    panel.plot(rows["C"], rows["cv_F1"], marker="o", markersize=8, linewidth=2,
               color=PLOT_BLUE, label="validation (ten folds)")
    panel.plot(rows["C"], rows["test_F1"], marker="o", markersize=8, linewidth=2,
               linestyle="--", color=PLOT_ORANGE, label="test")
    panel.axvline(1.0, color="grey", linestyle=":", linewidth=1, label="the default")
    panel.set_xscale("log")
    panel.set_title(family)
    panel.set_xlabel("C (log scale)")
panels[0].set_ylabel("F1")
panels[0].legend()
plt.show()

<div class="alert alert-block alert-warning">

**The prediction holds.** At `C=0.001` L1 and elastic net fall to
**0.7532**, the floor's F1, because every weight is zero; L2 only shrinks, and
keeps **0.8153**. From `C=1` on the three sit between **0.8493** and **0.8510**:
once the penalty is light, its strength and its shape barely matter.

</div>

**No penalty: is regularisation doing anything at all?**
`C=np.inf` removes the charge. The statsmodels fit already showed what that does
to the coefficients; the question here is whether it changes any label.
**Prediction: F1 does not move.**

__Step 11:__ Fit the default with and without its penalty.

In [ ]:
# C=np.inf removes the penalty; C=1 is the default it replaces.
penalty_rows = []
for value in [1.0, np.inf]:
    model = LogisticRegression(C=value, max_iter=5000)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                  for fold, fitted in zip(churn_prepared, fold_models)]
    penalty_rows.append({
        "configuration": "L2, C=1" if value == 1.0 else "no penalty, C=inf",
        "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                  for fold, fitted in zip(churn_prepared, fold_models)],
        "validation": validation, "cv_F1": np.mean(validation),
        "test_F1": f1_score(y_churn_test, clone(model).fit(
            X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)),
        "model": model,
    })
penalty_sweep = pd.DataFrame(penalty_rows)
print(penalty_sweep[["configuration", "cv_F1", "test_F1"]].round(4).to_string(index=False))

In [ ]:
# The sweep's winner joins the log.
best = penalty_sweep.loc[penalty_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

<div class="alert alert-block alert-warning">

**0.8494 with the default penalty and 0.8505 without it on the
folds; 0.8474 and 0.8451 on the test rows.** The two columns move by about a
thousandth, in opposite directions, so the prediction holds as far as the folds
can tell. Removing the penalty changed the coefficients a great deal, as the
statsmodels fit showed, and the labels hardly at all: at `C=1` the penalty is
already too light to decide who is called a positive.

</div>

**`class_weight`: should the rarer class count for more?** With
`'balanced'`, each class is weighted by the inverse of its frequency, so errors
on the smaller class cost the fitter proportionally more. It is the standard
answer to an imbalanced target.

Here the split is **2,416 to 1,584** and the majority class is the **positive**
one, so balancing takes weight *away* from the class F1 is computed on.
**Prediction: recall falls, and F1 with it, while precision rises.**

__Step 12:__ Reweight the classes and watch three metrics, not one.

In [ ]:
# Three metrics per value, so the trade between recall and precision is visible.
weight_rows = []
for weight in [None, "balanced"]:
    model = LogisticRegression(class_weight=weight, max_iter=5000)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"L2, C=1, class_weight={weight}",
           "class_weight": str(weight), "model": model}
    for name, metric in [("F1", f1_score), ("recall", recall_score),
                         ("precision", precision_score)]:
        row[f"cv_{name}"] = np.mean([metric(fold["y_score"], fitted.predict(fold["X_score"]))
                                     for fold, fitted in zip(churn_prepared, fold_models)])
        row[f"test_{name}"] = metric(y_churn_test, labels)
    weight_rows.append(row)
weight_sweep = pd.DataFrame(weight_rows)
print(weight_sweep.drop(columns="model").round(4).to_string(index=False))

In [ ]:
metrics = ["F1", "recall", "precision"]
positions = np.arange(len(metrics))
figure, panels = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for panel, prefix, title in zip(panels, ["cv", "test"], ["validation (ten folds)", "test"]):
    for offset, (colour, row) in enumerate(zip([PLOT_BLUE, PLOT_ORANGE],
                                               weight_sweep.itertuples())):
        panel.bar(positions + offset * 0.38,
                  [getattr(row, f"{prefix}_{name}") for name in metrics],
                  width=0.38, color=colour, label=f"class_weight={row.class_weight}")
    panel.set_xticks(positions + 0.19, metrics)
    panel.set_title(title)
panels[0].set_ylim(0.6, 0.9)
panels[0].set_ylabel("score")
panels[0].legend()
figure.suptitle("'balanced' is a trade, not a fix")
plt.show()

**The prediction holds in both columns.** On the folds `'balanced'`
lowers recall from **0.8890** to **0.8262** and F1 from **0.8494** to
**0.8330**, and raises precision from **0.8134** to **0.8402**. On the test rows
recall falls from **0.8965** to **0.8364** and F1 from **0.8474** to **0.8390**,
and precision rises from **0.8033** to **0.8417**. Weight moved from the
positives to the negatives, so the model says "yes" less often and is right more
often when it does.

That is a trade, not a failure. F1 is computed on the positive class, so it
prefers the unweighted model; a project that pays more for a false alarm than
for a missed positive would choose `'balanced'`. Choosing the metric is a
decision about the problem, and the metric then chooses the setting. Here F1
keeps the default, which the log already holds, so this step adds no row.

**`fit_intercept`: are the log-odds allowed a constant term?** The
intercept is the log-odds of an athlete at zero on every prepared column: at the
median of every numeric feature, which `RobustScaler` maps to zero, and at the
reference level of every category. Turn it off and that athlete is forced to
log-odds 0, a probability of 0.5, and the other coefficients absorb the
difference.

statsmodels gave `const` a p-value of **0.558**: the data cannot tell it from
zero. **Prediction: dropping it should cost nothing.**

__Step 13:__ Fit the default with and without the intercept.

In [ ]:
churn_intercept_rows = []
for value in [True, False]:
    model = LogisticRegression(fit_intercept=value, max_iter=5000)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                  for fold, fitted in zip(churn_prepared, fold_models)]
    churn_intercept_rows.append({
        "configuration": f"L2, C=1, fit_intercept={value}",
        "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                  for fold, fitted in zip(churn_prepared, fold_models)],
        "validation": validation, "cv_F1": np.mean(validation),
        "test_F1": f1_score(y_churn_test, clone(model).fit(
            X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)),
        "model": model,
    })
churn_intercept_sweep = pd.DataFrame(churn_intercept_rows)
print(churn_intercept_sweep[["configuration", "cv_F1", "test_F1"]].round(4).to_string(index=False))

In [ ]:
# The sweep's winner joins the log.
best = churn_intercept_sweep.loc[churn_intercept_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

**0.8494 with the intercept and 0.8503 without on the folds; 0.8474
and 0.8484 on the test rows.** The prediction holds: the other coefficients
absorb what the constant carried, and the difference is a thousandth in each
column.

Every row the steps added to the log is one step away from the
defaults, so combining those winners would be a guess at the best joint setting
from one-dimensional evidence. A grid search fits every combination instead, so
it can land on a pairing no single step visited.

__Step 14:__ Search every combination of the settings tested.

In [ ]:
# Two sub-grids: C=np.inf is the model with no penalty, so no strength or
# shape is left to tune.
churn_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(classification_recipe),
                      LogisticRegression(max_iter=5000)),
    [{"estimator": list(penalties.values()), "estimator__C": c_values,
      "estimator__class_weight": [None, "balanced"],
      "estimator__fit_intercept": [True, False]},
     {"estimator": [penalties["L2"]], "estimator__C": [np.inf],
      "estimator__class_weight": [None, "balanced"],
      "estimator__fit_intercept": [True, False]}],
    scoring="f1", cv=churn_folds, n_jobs=4,
)

In [ ]:
churn_search = clone(churn_template).fit(X_churn_train, y_churn_train)

In [ ]:
# The five best combinations, each beside the test F1 of its refit.
names = {model.l1_ratio: name for name, model in penalties.items()}
results = churn_search.cv_results_
ranking = pd.DataFrame({
    "penalty": [names[model.l1_ratio] if C != np.inf else "none"
                for model, C in zip(results["param_estimator"], results["param_estimator__C"])],
    "C": results["param_estimator__C"],
    "class_weight": [str(weight) for weight in results["param_estimator__class_weight"]],
    "fit_intercept": results["param_estimator__fit_intercept"],
    "cv_F1": results["mean_test_score"],
}).nlargest(5, "cv_F1")
ranking["test_F1"] = [
    f1_score(y_churn_test, clone(churn_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(
            X_churn_train, y_churn_train).predict(X_churn_test))
    for index in ranking.index]
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(4).to_string(), "\n")

# The winner's own score is the best of all of them on the same folds.
winner_churn = cross_validate(
    clone(churn_search.best_estimator_), X_churn_train, y_churn_train,
    cv=churn_folds, scoring="f1", return_train_score=True)
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "joint search winner",
                   "train": winner_churn["train_score"],
                   "validation": winner_churn["test_score"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "winner's own score", metric=f1_score, label="F1")

**The search picks L1 at `C=100`, without an intercept or class
weights**, at a validation F1 of **0.8511** and **0.8512** on the test rows. No
single step visited that pairing, and elastic net at the same setting ties it to
four decimals. All five leaders leave `class_weight` at its default, and the
whole top five sits far inside the spread of the folds.

The winner's own score is the best of 76 on the same ten folds, so it carries
the luck of being chosen. The next section estimates the search on rows that
took no part in the choice.

# <font color='#E8800A'>Nested cross-validation on the athletes</font> <a class="anchor" id="nested-athletes"></a>
[Back to TOC](#toc)

**Selection and estimation need different rows.** Nested
cross-validation repeats the *whole* search inside each of the course's ten
outer folds. The inner folds choose the settings from the outer training part
alone, and the outer validation part, which took no part in that choice, scores
the result. The mean over the ten outer folds estimates the procedure, choice
included, and the scoreboard publishes it first.

__Step 15:__ Estimate the athletes' search with nested cross-validation.

In [ ]:
churn_nested = cross_validate(
    churn_template, X_churn_train, y_churn_train, cv=churn_folds,
    scoring={"f1": "f1", "accuracy": "accuracy"},
    return_train_score=True,
)
churn_nested_rows = []
for metric in ("f1", "accuracy"):
    validation_values = churn_nested[f"test_{metric}"]
    churn_nested_rows.append({
        "metric": metric,
        "outer_train_mean": churn_nested[f"train_{metric}"].mean(),
        "outer_validation_mean": validation_values.mean(),
        "fold_sd": validation_values.std(ddof=1),
        "SEM": validation_values.std(ddof=1) / np.sqrt(len(validation_values)),
    })
print(pd.DataFrame(churn_nested_rows).round(4).to_string(index=False), "\n")

# The nested estimate joins the log, beside the same deployed model's test score.
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "joint search winner",
                   "train": churn_nested["train_f1"],
                   "validation": churn_nested["test_f1"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "nested estimate", metric=f1_score, label="F1")

**F1 0.8486 over the ten outer folds**, with a fold standard deviation
of 0.0176. This is the test the grid had to pass. The estimate sits below the
inner winner's 0.8511 and below the default's **0.8494**: part of the inner score
was the luck of being chosen among 76, and the search, estimated on rows it never
saw, does not beat leaving the settings alone. The outer training mean, 0.8583,
sits close to the validation mean, so the model is not overfitting its folds.

The default's estimate sits **0.0021** above its test F1 and the winner's
**0.0027** below it, so the two lean opposite ways. The assessment is the
repeated cross-validation; the test column is one more fold, drawn once.

# <font color='#E8800A'>Predicting a number: linear regression</font> <a class="anchor" id="linear"></a>
[Back to TOC](#toc)

The second half moves to a numeric target. The regression spine is
`cars4you`: 4,000 real used cars, with
`price` in euros as the target. A price is a continuous number, so linear
regression is the model the target type calls for.

__Step 16:__ Load the data, split it, and print what we are predicting.

In [ ]:
cars, regression_recipe = load_recipe(
    "../../data/interim/cars4you.csv",
    regression_recipe_log,
)
cars_inputs = cars.loc[:, regression_recipe["categorical"] + regression_recipe["numeric"]].copy()
y_cars = cars["price"]

cars_train, cars_test, y_cars_train, y_cars_test = train_test_split(
    cars_inputs, y_cars, test_size=0.2, random_state=random_state,
)
print(f"{len(cars_train)} train rows, {len(cars_test)} test rows,"
      f" {cars_inputs.shape[1]} columns before encoding")
print(cars_train.join(y_cars_train).corr(numeric_only=True)["price"]
      .round(3).sort_values().to_string())

`engineSize` correlates with price at **0.646** and `year` at
**0.491**: a bigger, newer car costs more. `mileage`
(**-0.422**), `km_per_year` (**-0.388**) and `mpg` (**-0.308**) run the other
way, and `paintQuality%`
(**-0.014**) and `previousOwners` (**0.004**) barely move with price at all.
Each of these looks at one column alone; a coefficient below holds the other
columns fixed, which is a different question.

There is no `stratify=` here: that argument needs classes, and a continuous
target has none.

__Step 17:__ Prepare the cars, and fix the folds every score is read on.

In [ ]:
cars_prep = FoldPreprocessor(regression_recipe)
cars_prep.fit(cars_train, y_cars_train)
X_cars_train = pd.DataFrame(
    cars_prep.transform(cars_train),
    columns=cars_prep.get_feature_names_out(),
    index=cars_train.index,
)
X_cars_test = pd.DataFrame(
    cars_prep.transform(cars_test),
    columns=cars_prep.get_feature_names_out(),
    index=cars_test.index,
)

# Ten folds of the training cars: every validation score below is a mean over them.
cars_folds = RepeatedKFold(n_splits=5, n_repeats=2, random_state=random_state)
cars_prepared = prepared_folds(
    FoldPreprocessor(regression_recipe), cars_folds, cars_train, y_cars_train, n_jobs=4)

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html'>sklearn.dummy.DummyRegressor(*, strategy='mean', constant=None, quantile=None)</a>

**Definition:**
A regressor that ignores the features and predicts one value for every row, by default the training mean: the floor any real model has to beat.

**Main Parameters:**
- `strategy` (default `'mean'`): the value predicted; `'median'`, `'quantile'` and `'constant'` are the alternatives.

**Main Attributes (after fitting):**
- `constant_`: the value predicted for every row.

**Main Methods:**
- `fit(X, y)`: reads only `y`.
- `predict(X)`
</div>

__Step 18:__ Score a model that predicts the training mean, and open the log.

In [ ]:
# The floor: every car is predicted at the training mean.
cars_floor_cv = cross_validate(
    DummyRegressor(strategy="mean"), cars_train, y_cars_train, cv=cars_folds,
    scoring="neg_mean_absolute_error", return_train_score=True)
cars_floor = DummyRegressor(strategy="mean").fit(cars_train, y_cars_train)

In [ ]:
cars_log = []
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "predict the mean",
                  "train": -cars_floor_cv["train_score"],
                  "validation": -cars_floor_cv["test_score"], "model": cars_floor},
       cars_test, y_cars_test, metric=mean_absolute_error, label="MAE")

The log works as it did on the athletes, in euros: the validation
column chooses, and the test column only records.

__Step 19:__ Fit a linear regression, score it on the same folds, and log it.

In [ ]:
# The model whose coefficients are read below.
linear = LinearRegression().fit(X_cars_train, y_cars_train)

linear_model = PreparedEstimator(...)
linear_cv = cross_validate(
    linear_model, cars_train, y_cars_train, cv=cars_folds,
    scoring="neg_mean_absolute_error", return_train_score=True)
linear_fitted = clone(linear_model).fit(cars_train, y_cars_train)

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "LinearRegression()",
                  "train": -linear_cv["train_score"],
                  "validation": ..., "model": linear_fitted},
       cars_test, y_cars_test, metric=mean_absolute_error, label="MAE")

**MAE 3038.16 EUR on the validation folds and 3102.24 on the test
rows**, against 7231.10 and 6843.46 for predicting the mean: the model more than
halves the error of knowing nothing, in euros you can say out loud to a client.
Its training MAE, 3000.31, sits close to its validation MAE, so the line is not
memorising its training rows.

Hold on to the 3038.16: a penalised linear model ends this notebook at
**2546.51** on the same folds, and almost all of the difference comes from the
target, not from a setting.

# <font color='#E8800A'>Reading coefficients without being fooled</font> <a class="anchor" id="coefficients"></a>
[Back to TOC](#toc)

A fitted linear model hands you one number per feature. Those numbers
are the reason to use this model at all, but they are easy to read wrongly. The
table below reads `linear`'s.

__Step 20:__ Read the coefficients the prepared matrix leaves.

In [ ]:
coefficients = pd.Series(
    linear.coef_, index=X_cars_train.columns
).sort_values(key=abs, ascending=False)
print(coefficients.head(8).round(2).to_string())
print("year is ranked", list(coefficients.index).index("year") + 1,
      "and mileage", list(coefficients.index).index("mileage") + 1,
      f"of {len(coefficients)}")
print(f"columns the solver separated: {linear.rank_}"
      f" of {X_cars_train.shape[1]}")

Three facts about the car lead the table, the ones a buyer would
name: `engineSize` at **44163.12**, `year` at **33813.55** and `mpg` at
**-33525.24**. `km_per_year` follows at **-10184.45**, and `mileage` has fallen
to tenth: `km_per_year` is mileage divided by the car's age, so the two columns
share their information and the fit splits it between them. Two indicators sit
high in the table for another reason, `fuelType_Electric` at **16547.37** and
`transmission_Other` at **-5997.72**, which the statsmodels table explains.

Every numeric column was scaled into $[0, 1]$, so one unit is the whole observed
span of that feature, with the other columns held fixed. An indicator's unit is
the step from the reference level. The solver separated **21 of 21** columns, so
no coefficient is standing in for another.

__Step 21:__ Fit the same model in statsmodels, which adds the statistical table scikit-learn does not produce.

`sm.OLS` is the least-squares counterpart of `sm.Logit`, and the
two tables follow the same order: first what scikit-learn holds for `linear`,
then the table statsmodels prints, which adds a standard error, a 95% confidence
interval and a p-value to each coefficient.

In [ ]:
sklearn_ols = pd.DataFrame({"coefficient": np.r_[linear.intercept_, linear.coef_]},
                           index=["const", *X_cars_train.columns])
sklearn_ols.round(2)

In [ ]:
ols_design = sm.add_constant(X_cars_train.astype(float), has_constant="add")
# HC3 standard errors stay valid when the error spread changes with the price.
ols_result = sm.OLS(y_cars_train.astype(float), ols_design).fit(cov_type="HC3")
print(ols_result.summary())

**The two libraries fit the same coefficients**; statsmodels prints
the large ones in scientific notation, so `engineSize`'s 44163.12 reads
**4.416e+04**. What statsmodels adds is precision: that coefficient's 95%
interval runs from **3.81e+04 to 5.02e+04**, large and tightly pinned.

**By the same 0.05 rule, six predictors are insignificant and could be
removed:** `Brand_BMW` (**0.828**), `Brand_Mercedes` (**0.057**), `Brand_VW`
(**0.198**), `transmission_Other` (**0.873**), `fuelType_Electric` (**0.502**)
and `fuelType_Other` (**0.361**). One training car carries each of
`transmission_Other` and `fuelType_Electric`, which is why theirs are the widest
intervals in the table. The intercept's **0.233** is not a predictor's: it is
the price of a car at every numeric minimum, and the next section tests dropping
it.

A small p-value says the data pin a coefficient away from zero. It does not say
the feature causes the price, or rank the features by importance.

# <font color='#E8800A'>Alternatives to least squares: Ridge, Lasso and Elastic Net</font> <a class="anchor" id="linear-params"></a>
[Back to TOC](#toc)

`LinearRegression` has nothing to tune. Its three standard
alternatives add a penalty on the size of the coefficients to the squared error
they minimise, so every coefficient has to earn its size.

<div class="alert alert-block alert-info">

| model | what it minimises | what the penalty does |
|---|---|---|
| [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) | $\lVert y - Xw \rVert_2^2 + \alpha \lVert w \rVert_2^2$ | shrinks every coefficient towards zero |
| [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) | $\frac{1}{2n}\lVert y - Xw \rVert_2^2 + \alpha \lVert w \rVert_1$ | shrinks, and sets the weakest exactly to zero |
| [`ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) | $\frac{1}{2n}\lVert y - Xw \rVert_2^2 + \alpha\rho \lVert w \rVert_1 + \frac{\alpha(1-\rho)}{2} \lVert w \rVert_2^2$ | both, mixed by `l1_ratio` $= \rho$ |

As `alpha` falls to zero each becomes ordinary least squares.

</div>

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html'>sklearn.linear_model.Ridge(alpha=1.0, *, fit_intercept=True, positive=False)</a>

**Definition:**
Least squares plus a penalty on the squared size of the weights (L2).

**Main Parameters:**
- `alpha` (default `1.0`): penalty strength; larger shrinks harder.
- `fit_intercept` (default `True`): whether the model gets a constant term.
- `positive` (default `False`): `True` forces every weight to be at least zero.

**Main Attributes (after fitting):**
- `coef_`: one weight per feature.
- `intercept_`: the constant term.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`
</div>

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html'>sklearn.linear_model.Lasso(alpha=1.0, *, fit_intercept=True, positive=False, max_iter=1000)</a>

**Definition:**
Least squares plus a penalty on the absolute size of the weights (L1), which sets the weakest exactly to zero.

**Main Parameters:**
- `alpha` (default `1.0`): penalty strength; larger shrinks harder.
- `fit_intercept` (default `True`): whether the model gets a constant term.
- `positive` (default `False`): `True` forces every weight to be at least zero.
- `max_iter` (default `1000`): the solver's iteration budget; raised here so every fit converges.

**Main Attributes (after fitting):**
- `coef_`: one weight per feature.
- `intercept_`: the constant term.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`
</div>

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html'>sklearn.linear_model.ElasticNet(alpha=1.0, *, l1_ratio=0.5, fit_intercept=True, positive=False, max_iter=1000)</a>

**Definition:**
Least squares plus a mix of the L1 and L2 penalties.

**Main Parameters:**
- `alpha` (default `1.0`): penalty strength; larger shrinks harder.
- `l1_ratio` (default `0.5`): the L1 share of the penalty; 1 is the Lasso penalty, 0 the Ridge one.
- `fit_intercept` (default `True`): whether the model gets a constant term.
- `positive` (default `False`): `True` forces every weight to be at least zero.
- `max_iter` (default `1000`): the solver's iteration budget; raised here so every fit converges.

**Main Attributes (after fitting):**
- `coef_`: one weight per feature.
- `intercept_`: the constant term.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`
</div>

As on the athletes, each step below changes **one** setting, prints
validation and test MAE side by side, and logs its best value. This half ends
with one grid search over every combination of the settings tested, and its
nested estimate.

**`alpha`: how much the model is charged for large weights.** At
$\alpha = 0$ nothing is charged and you have ordinary least squares; as
$\alpha$ grows the weights are pulled towards zero, and in the limit the model
predicts the mean for everything. That is a trade, not an improvement, so the
prediction is a **U-shape**: some shrinkage should help, and too much should
hurt.

The same `alpha` is not the same penalty in the three models: Lasso charges
absolute values rather than squares, and Lasso and ElasticNet average the error
over the rows first. **Prediction: each model's U bottoms out at a different
alpha, so one default cannot suit all three.**

__Step 22:__ Walk alpha across seven orders of magnitude for each alternative; at `alpha=1` each is at its defaults.

In [ ]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
# More solver iterations than Lasso's and ElasticNet's default 1000, so every fit
# converges; a fit that converged sooner is unchanged.
alternatives = [Ridge(), Lasso(max_iter=10000), ElasticNet(max_iter=10000)]

In [ ]:
alpha_rows = []
for default in alternatives:
    for value in alphas:
        model = clone(default).set_params(alpha=value)
        fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
        validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"]))
                      for fold, fitted in zip(cars_prepared, fold_models)]
        alpha_rows.append({
            "configuration": f"{type(model).__name__}(alpha={value:g})",
            "family": type(model).__name__, "alpha": value,
            "train": [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"]))
                      for fold, fitted in zip(cars_prepared, fold_models)],
            "validation": validation, "cv_MAE": np.mean(validation),
            "test_MAE": mean_absolute_error(y_cars_test, clone(model).fit(
                X_cars_train, y_cars_train).predict(X_cars_test)),
            "model": model,
        })
alpha_sweep = pd.DataFrame(alpha_rows)
alpha_sweep.pivot(index="alpha", columns="family", values=["cv_MAE", "test_MAE"]).round(2)

In [ ]:
# The sweep's winner joins the log.
best = alpha_sweep.loc[alpha_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

In [ ]:
figure, panels = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for panel, family in zip(panels, ["Ridge", "Lasso", "ElasticNet"]):
    rows = alpha_sweep[alpha_sweep["family"] == family]
    panel.plot(rows["alpha"], rows["cv_MAE"], marker="o", markersize=8, linewidth=2,
               color=PLOT_BLUE, label="validation (ten folds)")
    panel.plot(rows["alpha"], rows["test_MAE"], marker="o", markersize=8, linewidth=2,
               linestyle="--", color=PLOT_ORANGE, label="test")
    panel.axvline(1.0, color="grey", linestyle=":", linewidth=1, label="the default")
    panel.set_xscale("log")
    panel.set_title(family)
    panel.set_xlabel("alpha (log scale)")
panels[0].set_ylabel("MAE (EUR)")
panels[0].legend()
plt.show()

<div class="alert alert-block alert-warning">

**The prediction holds: each model bottoms out at its own alpha.**
Ridge's best is its default, **3019.33** at $\alpha = 1$, a null result that
eight values across seven orders of magnitude could have overturned. Lasso's best
is **3014.73** at $\alpha = 10$ and ElasticNet's **3016.95** at
$\alpha = 0.001$. ElasticNet's default, **5697.35**, sits far up its curve: it
weighs the squared half of its penalty against an error averaged over the rows.

</div>

In [ ]:
# The best of the three defaults joins the log; each step below changes one of its settings.
default_ridge = PreparedEstimator(FoldPreprocessor(regression_recipe), Ridge(alpha=1.0)).fit(
    cars_train, y_cars_train)
default_cars = cross_validate(
    PreparedEstimator(FoldPreprocessor(regression_recipe), Ridge(alpha=1.0)), cars_train, y_cars_train,
    cv=cars_folds, scoring="neg_mean_absolute_error", return_train_score=True)
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "Ridge(alpha=1), raw price",
                  "train": -default_cars["train_score"],
                  "validation": -default_cars["test_score"], "model": default_ridge},
       cars_test, y_cars_test, metric=mean_absolute_error, label="MAE")

**`fit_intercept`: is the model allowed a constant term?** The
intercept $\beta_0$ is what the model predicts when every feature sits at zero.
Turn it off and the fitted surface is forced through the origin, so a car at the
minimum of every numeric column, with every indicator at its reference level, is
predicted at **0 EUR**.

statsmodels gave the intercept a p-value of **0.233**: the data cannot tell it
from zero. **Prediction: dropping it should cost nothing.**

__Step 23:__ Fit `Ridge(alpha=1)` with and without the intercept.

In [ ]:
intercept_rows = []
for value in [True, False]:
    model = Ridge(alpha=1.0, fit_intercept=value)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
    validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"]))
                  for fold, fitted in zip(cars_prepared, fold_models)]
    intercept_rows.append({
        "configuration": f"Ridge(alpha=1, fit_intercept={value})",
        "train": [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"]))
                  for fold, fitted in zip(cars_prepared, fold_models)],
        "validation": validation, "cv_MAE": np.mean(validation),
        "test_MAE": mean_absolute_error(y_cars_test, clone(model).fit(
            X_cars_train, y_cars_train).predict(X_cars_test)),
        "model": model,
    })
intercept_sweep = pd.DataFrame(intercept_rows)
print(intercept_sweep[["configuration", "cv_MAE", "test_MAE"]].round(2).to_string(index=False))

In [ ]:
# The sweep's winner joins the log.
best = intercept_sweep.loc[intercept_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

**3019.33 with the intercept and 3018.13 without on the folds; 3073.50
and 3077.14 on the test rows.** The prediction holds: the other columns absorb
what the constant carried, and a few euros on errors of three thousand are far
inside the spread of the folds.

**`positive`: may a coefficient be negative?** Set it to `True`
and every weight is constrained to $\geq 0$, so no feature may push the price
down.

That is a claim we can check before running anything. The unconstrained fit
gives `mpg` **-33525.24** and `km_per_year` **-10184.45**, and statsmodels puts
both intervals wholly below zero. **Prediction: the constraint is false here, so
it should cost.**

__Step 24:__ Fit `Ridge(alpha=1)` with and without the sign constraint.

In [ ]:
positive_rows = []
for value in [False, True]:
    model = Ridge(alpha=1.0, positive=value)
    positive_rows.append({
        "configuration": f"Ridge(alpha=1, positive={value})",
        "cv_MAE": np.mean([
            mean_absolute_error(fold["y_score"], clone(model).fit(
                fold["X_fit"], fold["y_fit"]).predict(fold["X_score"]))
            for fold in cars_prepared]),
        "test_MAE": mean_absolute_error(y_cars_test, clone(model).fit(
            X_cars_train, y_cars_train).predict(X_cars_test)),
        "model": model,
    })
positive_sweep = pd.DataFrame(positive_rows)
print(positive_sweep[["configuration", "cv_MAE", "test_MAE"]].round(2).to_string(index=False))

**3019.33 unconstrained against 3418.97 constrained on the folds, and
3073.50 against 3540.86 on the test rows.** The constraint forces every negative
coefficient to zero, including the two the intervals had pinned below it, and the
model pays **399.64 EUR** on the folds for it.

`positive=True` is the right setting when a relationship cannot be negative, as
in many physical and cost models. Here the assumption is false, and the damage
is the size of the assumption. The default wins, and the log already holds it.

# <font color='#E8800A'>Transforming the target: log(price)</font> <a class="anchor" id="target"></a>
[Back to TOC](#toc)

Every setting has now been moved alone, and none beat Ridge's
default by more than 4.60 EUR on the folds. What is left to change is the target
itself: fit the same `Ridge(alpha=1)` to log(price) instead of price, and report
the error in euros.

__Step 25:__ Describe the target before deciding how to model it.

In [ ]:
print(f"training price skew {y_cars_train.skew():.3f} as it stands,"
      f" {np.log(y_cars_train).skew():.3f} after a log")
print(f"training price runs {y_cars_train.min():,.0f} to {y_cars_train.max():,.0f} EUR,"
      f" median {y_cars_train.median():,.0f}, mean {y_cars_train.mean():,.0f}")

**Skew 2.551, and -0.116 once logged.** Training prices run from
650 to 145,000 EUR around a median of 14,195. A few cars cost ten times the
median while none costs less than zero, and the log undoes almost exactly that
stretch.

It also matches what a price *is*. Depreciation is **multiplicative**: a car
worth twice as much loses roughly twice as many euros a year. The data therefore
look more like $\text{price} = e^{\beta_0 + \beta_1 x_1 + \dots}$ than like a
weighted sum, and taking the log of the target turns the first into the second:

$$\log(\text{price}) = \beta_0 + \beta_1 x_1 + \dots .$$

Predictions are exponentiated back to euros, so the MAE compares with every row
above; an exponentiated prediction estimates the median price, which suits MAE.
**Prediction: this should be worth more than every setting put together.**

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html'>sklearn.compose.TransformedTargetRegressor(regressor=None, *, func=None, inverse_func=None)</a>

**Definition:**
Wraps a regressor so that it fits a transformed target and predicts on the original scale.

**Main Parameters:**
- `regressor` (default `None`, which fits a `LinearRegression`): the model fitted to the transformed target.
- `func` (default `None`): applied to `y` before fitting; `None` leaves it unchanged.
- `inverse_func` (default `None`): applied to the predictions, back to the target's units.

**Main Attributes (after fitting):**
- `regressor_`: the fitted regressor inside.

**Main Methods:**
- `fit(X, y)`: transforms `y`, then fits the regressor.
- `predict(X)`: predicts, then applies `inverse_func`.
</div>

__Step 26:__ Fit the same ridge to log(price), and report euros.

In [ ]:
# With both functions None, TransformedTargetRegressor fits the raw price.
target_rows = []
for label, forward, inverse in [("raw price", None, None), ("log(price)", np.log, np.exp)]:
    model = TransformedTargetRegressor(regressor=Ridge(alpha=1.0), func=forward,
                                       inverse_func=inverse)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
    validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"]))
                  for fold, fitted in zip(cars_prepared, fold_models)]
    target_rows.append({
        "configuration": f"Ridge(alpha=1), {label}",
        "train": [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"]))
                  for fold, fitted in zip(cars_prepared, fold_models)],
        "validation": validation, "cv_MAE": np.mean(validation),
        "test_MAE": mean_absolute_error(y_cars_test, clone(model).fit(
            X_cars_train, y_cars_train).predict(X_cars_test)),
        "model": model,
    })
target_sweep = pd.DataFrame(target_rows)
print(target_sweep[["configuration", "cv_MAE", "test_MAE"]].round(2).to_string(index=False))

In [ ]:
# The sweep's winner joins the log.
best = target_sweep.loc[target_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

In [ ]:
# Each fold's ridge predicts its validation part, so every training car
# appears once per repeat.
figure, panels = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for panel, (label, row) in zip(panels, [("raw price", 0), ("log(price)", 1)]):
    ridge = target_sweep.loc[row, "model"]
    for fold in cars_prepared:
        fitted = clone(ridge).fit(fold["X_fit"], fold["y_fit"])
        panel.scatter(fold["y_score"], fitted.predict(fold["X_score"]), s=6,
                      alpha=0.2, color=PLOT_BLUE)
    panel.plot([y_cars_train.min(), y_cars_train.max()],
               [y_cars_train.min(), y_cars_train.max()],
               color=PLOT_ORANGE, linewidth=1)
    panel.set_title(f"ridge on {label}")
    panel.set_xlabel("actual price (EUR)")
panels[0].set_ylabel("validation prediction (EUR)")
plt.show()

<div class="alert alert-block alert-info">

**2555.51 against 3019.33 on the folds, and 2638.28 against 3073.50
on the test rows: 463.82 EUR, or 15.4 %, from changing what is predicted.** No
setting came close: Ridge's best `alpha` was its default, no tuned alternative
beat it by five euros, dropping the intercept moved the error by 1.20 EUR, and
the sign constraint cost 399.64 EUR. Both rows share folds and `Ridge(alpha=1)`,
so the transform is the one thing that differs.

The scatterplots, drawn from each fold's validation predictions, show why. On
the raw target the cloud bends below the diagonal as prices rise, and the
cheapest cars get predictions below zero: a straight line in euros cannot follow
a price that falls by a proportion. On the logged target the cloud follows the
diagonal across the range. The grid search that follows therefore searches the
target along with every setting.

</div>

As on the athletes, each single change in the log is one step from the
defaults, so one grid search fits every combination of the settings tested.

__Step 27:__ Search every combination of the settings tested.

In [ ]:
# Two sub-grids, one per target. With both functions None,
# TransformedTargetRegressor fits the raw price.
cars_grid = [
    {"estimator__regressor": alternatives,
     "estimator__regressor__alpha": alphas,
     "estimator__regressor__fit_intercept": [True, False],
     "estimator__regressor__positive": [True, False],
     "estimator__func": [forward], "estimator__inverse_func": [inverse]}
    for forward, inverse in ((None, None), (np.log, np.exp))
]
cars_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(regression_recipe),
                      TransformedTargetRegressor(regressor=Ridge())),
    cars_grid, scoring="neg_mean_absolute_error", cv=cars_folds, n_jobs=4,
)

In [ ]:
cars_search = clone(cars_template).fit(cars_train, y_cars_train)

In [ ]:
# The five best combinations, each beside the test MAE of its refit.
results = cars_search.cv_results_
ranking = pd.DataFrame({
    "family": [type(model).__name__ for model in results["param_estimator__regressor"]],
    "alpha": results["param_estimator__regressor__alpha"],
    "fit_intercept": results["param_estimator__regressor__fit_intercept"],
    "positive": results["param_estimator__regressor__positive"],
    "target": ["log(price)" if func is np.log else "raw price"
               for func in results["param_estimator__func"]],
    "cv_MAE": -results["mean_test_score"],
}).nsmallest(5, "cv_MAE")
ranking["test_MAE"] = [
    mean_absolute_error(y_cars_test, clone(cars_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(
            cars_train, y_cars_train).predict(cars_test))
    for index in ranking.index]
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(2).to_string(), "\n")

# The winner's own score is the best of all of them on the same folds.
winner_cars = cross_validate(
    clone(cars_search.best_estimator_), cars_train, y_cars_train,
    cv=cars_folds, scoring="neg_mean_absolute_error", return_train_score=True)
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "joint search winner",
                  "train": -winner_cars["train_score"],
                  "validation": -winner_cars["test_score"],
                  "model": cars_search.best_estimator_},
       cars_test, y_cars_test, "winner's own score", metric=mean_absolute_error, label="MAE")

**The search picks `Lasso(alpha=0.0001)` on log(price)**, with the
intercept on and no sign constraint, at a validation MAE of **2546.05 EUR** and
**2652.89** on the test rows. No single step visited that pairing. ElasticNet
(**2547.10**) and Ridge (**2547.91**) follow within two euros, far inside the
spread of the folds: once the target is logged, the choice of penalty barely
matters.

The winner's own score is the best of 192 on the same ten folds, so it carries
the luck of being chosen. The next section estimates the search on rows that
took no part in the choice.

# <font color='#E8800A'>Nested cross-validation on the cars</font> <a class="anchor" id="nested-cars"></a>
[Back to TOC](#toc)

As on the athletes, the whole search runs again inside each of the
ten outer folds, and each outer validation part scores what that search chose.

__Step 28:__ Estimate the cars' search with nested cross-validation.

In [ ]:
cars_nested = cross_validate(
    cars_template, cars_train, y_cars_train, cv=cars_folds,
    scoring={"mae": "neg_mean_absolute_error",
             "rmse": "neg_root_mean_squared_error"},
    return_train_score=True,
)
cars_nested_rows = []
for metric in ("mae", "rmse"):
    # The scorers return negated errors, so the sign is flipped back.
    validation_values = -cars_nested[f"test_{metric}"]
    cars_nested_rows.append({
        "metric": metric,
        "outer_train_mean": -cars_nested[f"train_{metric}"].mean(),
        "outer_validation_mean": validation_values.mean(),
        "fold_sd": validation_values.std(ddof=1),
        "SEM": validation_values.std(ddof=1) / np.sqrt(len(validation_values)),
    })
print(pd.DataFrame(cars_nested_rows).round(2).to_string(index=False), "\n")

# The nested estimate joins the log, beside the same deployed model's test score.
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "joint search winner",
                  "train": -cars_nested["train_mae"],
                  "validation": -cars_nested["test_mae"],
                  "model": cars_search.best_estimator_},
       cars_test, y_cars_test, "nested estimate", metric=mean_absolute_error, label="MAE")

**MAE 2546.51 EUR over the ten outer folds**, with a fold standard
deviation of 187.03 EUR, within a euro of the inner winner's 2546.05: choosing
among 192 cost nothing measurable. The outer training mean, 2514.86 EUR, sits
close to the validation mean, so the model is not overfitting its folds.

Here the test rows are harder than the average fold: from the linear model on,
every estimate sits below its test MAE, by **54.16 EUR** for Ridge's default and
**106.38 EUR** nested.

# <font color='#E8800A'>Two numbers we stand behind</font> <a class="anchor" id="two-numbers"></a>
[Back to TOC](#toc)

Each half ended with the nested estimate of its search. The scoreboard
publishes it beside the held-out score, the test column of each winner's log row.

__Step 29:__ Score each winner and each default once on the test rows.

In [ ]:
chosen_logistic = churn_search.best_estimator_
tuned_f1 = f1_score(y_churn_test, chosen_logistic.predict(X_churn_test))
default_logistic = PreparedEstimator(FoldPreprocessor(classification_recipe),
                                     LogisticRegression(max_iter=5000)).fit(X_churn_train, y_churn_train)
default_f1 = f1_score(y_churn_test, default_logistic.predict(X_churn_test))
chosen_cars = cars_search.best_estimator_
tuned_mae = mean_absolute_error(y_cars_test, chosen_cars.predict(cars_test))
default_mae = mean_absolute_error(y_cars_test, default_ridge.predict(cars_test))

**On the 800 held-out athletes the winner scores 0.8512 against
0.8474 for the default, and on the 800 held-out cars 2652.89 EUR against
3073.50** for `Ridge(alpha=1)` on the raw price, 420.61 EUR better. The
athletes' gain is the test column disagreeing with the nested estimate, which
put the search below the default; one draw of 800 rows does not overturn ten
outer folds.

Neither number is adjusted. The configuration was chosen on the training folds
before the test rows were opened, and choosing again after reading them would
turn the test set into one more fold of selection.

__Step 30:__ Record the two results, save the two winners, and open the course scoreboard.

In [ ]:
# Each row carries two numbers: the nested cross-validated mean first, then
# the winner's held-out score.
results_dir = "results"

print(f"{'':10s}{'F1':>9s}{'ROC-AUC':>10s}{'balanced acc':>14s}")
for name, model in (("default", default_logistic), ("tuned", chosen_logistic)):
    predicted = model.predict(X_churn_test)
    print(f"{name:10s}{f1_score(y_churn_test, predicted):9.4f}"
          f"{roc_auc_score(y_churn_test, model.predict_proba(X_churn_test)[:, 1]):10.4f}"
          f"{balanced_accuracy_score(y_churn_test, predicted):14.4f}")

# The defaults were measured on these same folds, so they are the baselines.
default_churn_cv = c_sweep.set_index("configuration").loc["L2, C=1", "cv_F1"]
default_ridge_cv = -default_cars["test_score"].mean()
print(f"defaults over the outer folds: F1 {default_churn_cv:.4f},"
      f" MAE {default_ridge_cv:.2f} EUR")

churn_board = Scoreboard("champions outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_churn_train.index.tolist(), y_churn_test.index.tolist(),
    outer_folds=[X_churn_train.index[rows].tolist()
                 for _, rows in churn_folds.split(X_churn_train, y_churn_train)]),
    floor=round(f1_score(y_churn_test, DummyClassifier(strategy="most_frequent").fit(
        X_churn_train, y_churn_train).predict(X_churn_test)), 4))
churn_board.record(6, "logistic regression (L1, C=100, no intercept)",
                   round(churn_nested["test_f1"].mean(), 4), round(tuned_f1, 4))
churn_board.save(
    results_dir, 6, "classification", baseline_cv=round(default_churn_cv, 4),
    baseline_test=round(default_f1, 4),
)

cars_board = Scoreboard("cars4you resale", "MAE (EUR)", context=result_context(
    "../../data/interim/cars4you.csv", regression_recipe_log,
    y_cars_train.index.tolist(), y_cars_test.index.tolist(),
    outer_folds=[cars_train.index[rows].tolist() for _, rows in cars_folds.split(cars_train)]),
    floor=round(mean_absolute_error(y_cars_test, DummyRegressor(strategy="mean").fit(
        cars_train, y_cars_train).predict(cars_test)), 4))
cars_board.record(6, "Lasso(alpha=0.0001), log(price)",
                  round(-cars_nested["test_mae"].mean(), 4), round(tuned_mae, 4))
cars_board.save(
    results_dir, 6, "regression", baseline_cv=round(default_ridge_cv, 4),
    baseline_test=round(default_mae, 4),
)

# The fitted winners are saved beside their results, so a later week can reload
# either one if it ends the course as the best model. Loading needs
# preprocessing.py on the path, as here.
joblib.dump(chosen_logistic, f"{results_dir}/week_06_classification_model.joblib")
joblib.dump(chosen_cars, f"{results_dir}/week_06_regression_model.joblib")

print()
print(churn_board.table())
print()
print(cars_board.table())

<div class="alert alert-block alert-success">

**Two baselines, and a scoreboard that now exists.** Each row
carries two numbers, cross-validated first. Logistic regression predicts the
champions outcome at F1 **0.8486** over the ten outer folds and **0.8512** held
out, against a **0.7529** floor. A penalised linear model on `log(price)`
predicts resale at MAE **2546.51 EUR** over the outer folds and **2652.89 EUR**
held out, against a **6843.46** floor. Both rows are saved to `results/`, with
the fitted winners beside them, so every later week compares against them, and
the course's best model can be reloaded if it turns out to be one of these.

The halves end in opposite places. On the athletes the nested estimate sits
below the default's 0.8494 while the test rows favour the winner, 0.8512 against
0.8474, and both gaps are far inside one fold standard deviation. On the cars
the search beats the raw-target default in both columns, 3019.33 and 3073.50,
almost entirely through the log. What tuning is worth is a fact about the data,
and measuring it can return nothing.

**A later model is not obliged to win.** When a more complicated model only
matches a simpler one, the simpler one is the better choice: it is faster,
easier to explain, and there is less of it to go wrong.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **The type of the target picks the model, before any fitting**, and the
   impossible predictions make that a rule rather than a convention.
2. **A coefficient is read in the unit its prepared column carries, and trusted
   once you know its interval.** statsmodels fits the same coefficients as
   scikit-learn, and its standard errors exposed `No coach_True`, a column one
   athlete carries.
3. **In a project, the grid over every combination is the default.** This
   notebook moved one setting at a time so that each could be discussed, and the
   steps only logged what each setting did. The grid chose, and on both tasks it
   chose a pairing no single step visited: L1 at `C=100` without an intercept,
   and `Lasso(alpha=0.0001)` on the log target.
4. **Selection and estimation use different rows.** Inner folds choose, nested
   cross-validation estimates the choice, and the log keeps every estimate
   beside a test score that played no part in it. The nested estimate is what
   says whether the search was worth running.
5. **What a parameter is worth is a property of the dataset.** On the athletes
   the search, estimated by nested cross-validation, did not beat the default;
   on the cars the target paid 463.82 EUR and no hyperparameter did. Both
   results are published.
6. **Report against a floor and in more than one metric.**

Exit ticket: in one sentence, why can a linear model not be used to
predict a probability, and why is that still true when its predictions happen to
land inside $[0, 1]$?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- James, G., Witten, D., Hastie, T. & Tibshirani, R. *An Introduction to Statistical Learning*, Ch. 3 (linear regression) and Ch. 4.3 (why not linear regression for a classification target).
- Hosmer, D.W., Lemeshow, S. & Sturdivant, R.X. (2013). *Applied Logistic Regression*, 3rd ed. Wiley, Ch. 1-3 (the logit, and reading odds ratios).
- Harrell, F.E. (2015). *Regression Modeling Strategies*, 2nd ed. Springer, § 10.1 (why a transformed target is a modelling decision, not a trick).
- statsmodels API, [OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html) and [Logit](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html).
- scikit-learn User Guide, [Linear models](https://scikit-learn.org/stable/modules/linear_model.html) · [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) · [`TransformedTargetRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).